# Build the Neural Network
Le reti neurali sono composte da layer che svolgono delle operazioni sui dati. Il namespace `torch.nn` ci fornisce tutto quello che serve per costruire la nostra rete.
Ogni modulo in PyTorch è sottoclasse di `nn.Module` e una rete neurale è anch'essa un modulo costituito da altri moduli (layers).

Costruiremo una rete neurale per classificare le immagini nel dataset FashionMNIST

In [ ]:
import os
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

Vogliamo allenare il nostro modello sulla GPU, se disponibile

In [ ]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

## Define the class
Iniziamo a definire la nostra rete facendola sottoclasse di `nn.Module` e inizializzando i layer in `__init__`. Inoltre ogni sottoclasse di `nn.Module` deve implementare le operazione da fare sui dati di input nella funzione `forward`

In [ ]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28 * 28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

Adesso possiamo creare un'istanza di questa rete neurale e muoverla sulla GPU per stampare la sua struttura

In [ ]:
model = NeuralNetwork().to(device)
print(model)

Per utilizzare il modello gli dobbiamo passare dei dati in input, questo eseguirà la funzione `forward`.
Chiamare il modello sui dati ritornerà un tensor a due dimensioni:
- dim = 0 Indica quale esempio della batch stiamo guardando
- dim = 1 Indica i 10 valori previsti per quell'esempio, ogni colonna rappresenta una classe.

Creiamo una finta immagine e la passiamo alla rete per vedere cosa restituisce, ovvero un tensor da 10 elementi contente dei numeri, traformiamo con Softmax questi numeri in probabilità e poi prendiamo la classe con la probabilità più alta.

In [ ]:
X = torch.rand(1, 28, 28, device=device)
logits = model(X)
pred_probab = nn.Softmax(dim=1)(logits)
y_pred = pred_probab.argmax(1)
print(f"Predicted class is {y_pred}")

## Model Layers
Vediamo adesso cosa fanno nello specifico i layer scritti prima all'interno della rete.
Per aiutarci prendiamo un batch da 3 immagini casuali per vedere cosa succede

In [ ]:
input_image = torch.rand(3, 28, 28)
print(input_image.size())

### nn.Flatten
Questo layer converte ogni immagine 2D 28x28 in un array contiguo di 784 pixels

In [ ]:
flatten = nn.Flatten()
flat_image = flatten(input_image)
print(flat_image.size())

### nn.Linear
Questo layer applica una funzione lineare all'input utilizzando i pesi memorizzati nel modello, in questo caso quindi schiaccia i 784 valori in 20 di output effettuando calcoli.

In [ ]:
layer1 = nn.Linear(in_features=28 * 28, out_features=20)
hidden1 = layer1(flat_image)
print(hidden1.size())

### nn.ReLU
Questo layer di attivazione non lineare è quello che crea i complessi collegamenti tra input e output del modello. Vengono applicati dopo una trasformazione lineare per introdurre appunto non linearità aiutando il modello ad imparare un vasto numero di fenomeni.

In [ ]:
print(f"Before ReLU: {hidden1}\n\n")
hidden1 = nn.ReLU()(hidden1)
print(f"After ReLU: {hidden1}")

### nn.Sequential
Consiste in un container ordinato di moduli, i dati vengono passati attraverso tutti i moduli nell'ordine definito. Possiamo utilizzarli per creare al volo una rete

In [ ]:
seq_modules = nn.Sequential(
    flatten,
    layer1,
    nn.ReLU(),
    nn.Linear(20, 10)
)
input_image = torch.rand(3, 28, 28)
logits = seq_modules(input_image)

### nn.Softmax
L'ultimo layer della rete ritorna i logits, ovvero i valori in $[-\infty, \infty]$ che vengono passati al modulo `nn.Softmax`. I logits vengono scalati in valori [0, 1] che rappresentano la probabilità predetta dal modello per ogni classe. Il parametro dim indica la dimensione sulla quale i valori sommati dovranno dare come risultato 1.

Stiamo quindi trasformando i valori ottenuti in probabilità, per ogni esempio avremo una certa probabilità di appartenenza ad ogni classe

In [ ]:
softmax = nn.Softmax(dim=1)
pred_probab = softmax(logits)

## Model Parameters
Molti layer all'interno della rete neurale sono parametrizzati ovveri hanno dei pesi associati che vengono ottimizzati durante l'allenamento. Quando una rete è sottoclasse di `nn.Module` traccia in automatico tutti i campi definiti nel modello e rende tutti i parametri accessibili utilizzando `parameters()` o `named_parameters()`

Proviamo ad iterare su ogni parametro e printare la sua grandezza e valore

In [ ]:
print(f"Model structure: {model}\n\n")

for name, param in model.named_parameters():
    print(f"Layer: {name} | Size: {param.size()} | Values: {param[:2]}\n")